## Project
Using Time Series Analysis To Forecast Future Vacancy Levels 

## Hypothesis
If we examine historical vacancy data with data science techniques,

we can identify key patterns that occur over time and contribute to accurate forecasting, 

to drive actionable insights on labour market tightness for monetary policy makers. 

Why this matters? 
For example: higher vacancy volumes -> lower unemployment -> higher inflation -> interest rate measures 

## Task 
with time estimates (h)

1. Automate scraping of files (0.5)
2. Consolidate data (0.5)
3. Plot data to highlight patterns and revisions (0.5)
4. Build a simple model to forecast vacancy levels (1)
5. Comment on evaluation, recommendations, and next steps. (0.5)

All code, including any helper modules, has been included in this notebook to keep it readable for the purpose of this time assessed task.  

# 1. Import libraries and data

o Automate the download (or scraping) of the CSV files on the ONS website with a subset of at least 20 

In [ ]:
# Import libraries

from pathlib import Path
from typing import List
from urllib.parse import urljoin
import random
import time
import requests
from bs4 import BeautifulSoup

In [ ]:
# Scrape data from the web

PREV_URL = "https://www.ons.gov.uk/employmentandlabourmarket/peopleinwork/employmentandemployeetypes/timeseries/ap2y/lms/previous"
RAW_DIR  = (Path.cwd() / "../data/raw").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)  
USER_AGENT = "FVL-Interview-Task/1.0 (+https://github.com/<seleklekterek>/fvl-project; contact:<197108180+seleklekterek@users.noreply.github.com>)"

# scrape and download functions

def scrape_ap2y_csv_links(page_url: str) -> List[str]:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    r = s.get(page_url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    urls = [
        urljoin(page_url, a["href"])
        for a in soup.select("a[href]")
        if "format=csv" in a["href"].lower()
    ]
    seen, out = set(), []
    for u in urls:                            
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def download_csvs(urls: List[str], 
                  raw_dir: Path, 
                  limit: int = 20, 
                  delay_s: float = 0.75, # amended and added below to respect ONS-rate limits
                  max_retries: int = 6,
                  backoff_base: float = 2.0
                  ):
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    raw_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for idx, url in enumerate(urls, start=1):
        if len(saved) >= limit:
            break
        dst = raw_dir / f"AP2Y_prev_{len(saved)+1:02d}.csv"

        if dst.exists() and dst.stat().st_size > 0:
            saved.append(dst)
            continue

        for attempt in range(max_retries):
            try:
                resp = s.get(url, timeout=60, allow_redirects=True)
                if resp.status_code == 429:                          # to explicitly handle rate limiting
                    ra = resp.headers.get("Retry-After")  
                    if ra:
                        wait = float(ra) + random.uniform(0.2, 0.8)  # add jitter 
                    else:
                        wait = (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                    time.sleep(wait)
                    continue
                resp.raise_for_status()

                dst.write_bytes(resp.content)
                if dst.stat().st_size > 0:
                    saved.append(dst)
                    time.sleep(delay_s + random.uniform(0, 0.5))
                break

            except requests.RequestException as exc:
                wait = (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                time.sleep(wait)
                if attempt == max_retries - 1:
                    print(f"[warn] {url} -> {exc}")
            except Exception as exc:
                print(f"[warn] {url} -> {exc}")
                break
    
    return saved

# run scrape and download
urls = scrape_ap2y_csv_links(PREV_URL)
downloaded = download_csvs(urls, RAW_DIR, limit=20)
len(downloaded), [p.name for p in downloaded[:5]]

# 2. Prepare data

o Consolidate the data into a single, structured format suitable for analysis by cleaning and
aligning the time series across vintages.

o Focus only the monthly series of each file.

o Ensure that each value can be attributed to both its observation date and the vintage date
(i.e. when the value was published).

o Hint: each file has metadata stored at the top of the file, which is useful to capture, especially the vintage (release) date.


# 3. Plot data 

o Plot how the vacancy estimates for a given month have changed across different vintages. 

o Highlight any patterns or revisions that occur over time.

# 4. Forecast data 

o Build a simple model to forecast future vacancy levels

# 5. Evaluation and recommendations

o Outline in words how you would evaluate the forecast performance while thinking about
how the data can change per vintage.

o What next steps would you take to take this analysis to the next level?